# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import glob
import graphical_sampling as gs
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm
from package_sampling.utils import inclusion_probabilities

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "XPC_SERVICE_NAME" redefined by R and overriding existing variable. Current: "application.com.jetbrains.pycharm.1003326.4172874", R: "0"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpk1AVhG", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV"
  warnings.warn(


# Loading and Determining Population

In [3]:
DATA_DIR = "populations"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))

coords_dict = {}
probs_dict = {}

for fp in csv_paths:
    name = os.path.splitext(os.path.basename(fp))[0]
    data = np.loadtxt(fp, delimiter=",", skiprows=1)
    coords = data[:, :2]
    probs  = data[:, -1]

    coord_name, prob_name, *rest = name.split("_")
    coord_name = 'cluster' if coord_name == 'clust' else coord_name
    prob_name = 'equal' if prob_name == 'eq' else 'unequal'

    coords_dict[coord_name] = coords
    probs_dict[coord_name] = probs_dict.get(coord_name, {})
    probs_dict[coord_name][prob_name] = probs

print(coords_dict.keys())
print(probs_dict.keys())
print(probs_dict['random'].keys())

dict_keys(['swiss', 'RegularPop1000', 'AggregatedPop1027', 'cluster', 'meuse', 'random', 'grid'])
dict_keys(['swiss', 'RegularPop1000', 'AggregatedPop1027', 'cluster', 'meuse', 'random', 'grid'])
dict_keys(['equal', 'unequal'])


In [4]:
N = 100
n = 5
coords = coords_dict['random']
probs = probs_dict['random']['unequal']
modified_probs = inclusion_probabilities(probs, n=n)
pop = gs.Population(coords, modified_probs)

# Building Initial Designs

In [5]:
orders = [
    # "lexico-yx",
    # "lexico-xy",
    # "random",
    # "angle_0",
    # "distance_0",
    "projection",
    # "center",
    "spiral",
    # "max",
    # "snake",
    # "hilbert",
]

In [6]:
initial_designs = []
combines = list(itertools.product(orders, orders))
num_trials = 2
for units_order, zones_order in tqdm(combines, desc="Generating initial designs", total=len(combines), unit="orders"):
    best = None
    best_score = np.inf
    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=(2, 2),
            zone_builder='sweep',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    initial_designs.append(gs.NewDesign(best))

    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=(1, 1),
            zone_builder='sweep',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    initial_designs.append(gs.NewDesign(best))


Generating initial designs:   0%|          | 0/4 [00:00<?, ?orders/s]/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpf2SRNE"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpdJUaQT"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" re

In [7]:
for design in initial_designs:
    print(design.kmeans.all_samples.shape, design.kmeans.expected_moran_score())

(114, 5) -0.25371075423144557
(114, 5) -0.25371075423144557
(115, 5) -0.2690932662717034
(115, 5) -0.2690932662717034
(115, 5) -0.23094133780997056
(100, 5) -0.2533234414459326
(114, 5) -0.2724446799431903
(114, 5) -0.2724446799431903


# Run Astar

In [9]:
moran_criteria = gs.criteria.MoranCriteria()

In [10]:
astar = gs.search.AStar(
    initial_designs,
    moran_criteria
)

best initial criteria value -0.2724446799431903


In [11]:
astar.run(
    max_iterations = 1000,
    num_new_nodes = 10,
    max_open_set_size = 1000,

    n_clusters_to_change_order_zone = 0,
    n_changes_in_order_of_zones = 0,

    n_clusters_to_change_order_units = 1,
    n_zones_to_change_order_units = 1,
    n_changes_in_order_of_units = 1,

    n_jobs=-1
)


parent node: -0.2724446799431903
child node: -0.27140730784319894
child node: -0.27232442330721374
child node: -0.2665521114222189
child node: -0.27506461116434916

New best criteria value: -0.27506461116434916

child node: -0.27242697330501287
child node: -0.27427793563171393
child node: -0.27332776562036

parent node: -0.27506461116434916
child node: -0.2780501596833269

New best criteria value: -0.2780501596833269

child node: -0.2756193143373902
child node: -0.27612377638877805
child node: -0.25853359783455354
child node: -0.2786009130156695

New best criteria value: -0.2786009130156695

child node: -0.269843262787166

parent node: -0.2786009130156695
child node: -0.2800650380624097

New best criteria value: -0.2800650380624097

child node: -0.2762443383598493
child node: -0.2784291177886824
child node: -0.2803646078760747

New best criteria value: -0.2803646078760747

child node: -0.27959764590894565
child node: -0.2796280660119709
child node: -0.2803646078760747

parent node: -0

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpVModjw"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpVModjw", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpa1QrXg"
  warnings.warn(


child node: -0.27529105045312763
child node: -0.2805064665806506
child node: -0.28010827355373713
child node: -0.28010827355373713
child node: -0.27914051349730656
child node: -0.2816038232620004
child node: -0.2846975888871589

New best criteria value: -0.2846975888871589

child node: -0.2820170320163213
child node: -0.2799245608999526

parent node: -0.2846975888871589
child node: -0.2845883764877458
child node: -0.2838936881512641
child node: -0.28209544088157373
child node: -0.2800701273459505
child node: -0.28201462502226504
child node: -0.285002033519316

New best criteria value: -0.285002033519316

child node: -0.285002033519316
child node: -0.28456889724403794

parent node: -0.285002033519316
child node: -0.28558791618267754

New best criteria value: -0.28558791618267754

child node: -0.2827747046403888
child node: -0.2829874125194541
child node: -0.28604690206883193

New best criteria value: -0.28604690206883193

child node: -0.285075598936138
child node: -0.28082074077249486
c

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpoAMcYj"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p

child node: -0.28566741175777294
child node: -0.29238562840739973
child node: -0.292450686485854
child node: -0.29290552393535824
child node: -0.293084955265449

New best criteria value: -0.293084955265449

child node: -0.2924745885671074
child node: -0.29262909341709326

parent node: -0.293084955265449


...

child node: -0.29141202031797364
child node: -0.28764521894875866
child node: -0.29140073041905906
child node: -0.28688527671057346
child node: -0.2926686231923797
child node: -0.2929773633232741
child node: -0.29275443648852845
child node: -0.2922795932970527
child node: -0.29367225977519157

New best criteria value: -0.29367225977519157


parent node: -0.29367225977519157
child node: -0.2949944468296255

New best criteria value: -0.2949944468296255

child node: -0.2930862020522404
child node: -0.29053660839926515
child node: -0.2945816320965265
child node: -0.29492140956270907

parent node: -0.2949944468296255
child node: -0.29021261846349544
child node: -0.2928352776611415
child node: -0.29508742494838563

New best criteria value: -0.29508742494838563

child node: -0.2954199616217375

New best criteria value: -0.2954199616217375

child node: -0.2916638711154571
child node: -0.29424329260838494
child node: -0.29521531582897825
child node: -0.29086383826033424
child node: -0.278927461

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp9Nw5LI"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmps4GKS0"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders

child node: -0.2894167093794687
child node: -0.29597530257762034
child node: -0.29534228695329773
child node: -0.2930183117928573
child node: -0.29723674730130395

New best criteria value: -0.29723674730130395


parent node: -0.29723674730130395
child node: -0.2916604845307184
child node: -0.2934690172636603
child node: -0.29573549800878984
child node: -0.2967205281108561

parent node: -0.2967205281108561
child node: -0.29442986324856124
child node: -0.2956320645133367
child node: -0.2906850975816903
child node: -0.2965086222280396
child node: -0.29634620812810514
child node: -0.29603150026709607
child node: -0.29327929830529204

parent node: -0.296541088454285
child node: -0.297180414957207
child node: -0.2935289331534144
child node: -0.29394527585789565
child node: -0.2967817211115055
child node: -0.29277217214239726
child node: -0.2932132970925618
child node: -0.29081641442758016
child node: -0.29057303791683753
child node: -0.2960979177340138

parent node: -0.297180414957207
child 

R callback write-console: 
  
R callback write-console: 
  


KeyboardInterrupt: 

In [12]:
astar.best_criteria_value

-0.29723674730130395

In [13]:
astar.best_design.kmeans.score_summary_df()

,expected,std
measure,,
density,-0.147039,0.172205
moran,-0.297237,0.106668
local_balance,0.658284,1.621211
voronoi,0.137078,0.073595


In [14]:
np.mean(np.abs(astar.best_design.kmeans.fips - probs))

np.float64(0.03952947461863138)

In [15]:
astar.best_design.kmeans.all_samples_probs.sum()

np.float64(1.000000000002)

# Run Bees

In [16]:
bee = gs.search.Bees(
    initial_designs,
    gs.criteria.MoranCriteria(),
    colony_size=20,
    limit=50,
)

Evaluating initial designs...
ABC initialized - Best initial criteria value: -0.2724446799431903


In [17]:
bee.run(
    max_iterations = 1000,

    n_clusters_to_change_order_zone = 0,
    n_changes_in_order_of_zones = 0,

    n_clusters_to_change_order_units = 1,
    n_zones_to_change_order_units = 1,
    n_changes_in_order_of_units = 1,

    n_jobs=-1
)

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpeTee5b"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpdy4fbV", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpHsCss4"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders


Starting ABC with 20 food sources
Colony size: 20, Limit: 50

Iteration 1/1000
NEW BEST at iteration 1
Criteria value: -0.27428648
  Best: -0.274286, Avg: -0.262854

Iteration 2/1000
  Best: -0.274286, Avg: -0.264239

Iteration 3/1000
  Best: -0.274286, Avg: -0.264701

Iteration 4/1000
  Best: -0.274286, Avg: -0.264897

Iteration 5/1000
  Best: -0.274286, Avg: -0.264992

Iteration 6/1000
NEW BEST at iteration 6
Criteria value: -0.27522368
  Best: -0.275224, Avg: -0.265402

Iteration 7/1000
NEW BEST at iteration 7
Criteria value: -0.27727203
  Best: -0.277272, Avg: -0.265683

Iteration 8/1000
  Best: -0.277272, Avg: -0.266011

Iteration 9/1000
  Best: -0.277272, Avg: -0.267234

Iteration 10/1000
  Best: -0.277272, Avg: -0.267314

Iteration 11/1000
  Best: -0.277272, Avg: -0.267957

Iteration 12/1000
NEW BEST at iteration 12
Criteria value: -0.27943575
  Best: -0.279436, Avg: -0.268396

Iteration 13/1000
  Best: -0.279436, Avg: -0.268901

Iteration 14/1000
  Best: -0.279436, Avg: -0.268

R callback write-console: 
  


KeyboardInterrupt: 